# Sentinel Health ? Chronic Kidney Disease

End-to-end exploration and classification using the CKD ARFF files in this project. Positive class: `ckd`; negative class: `notckd`. The notebook inspects the source, handles missingness, reviews outliers, engineers safe features, compares 30 preprocessing/model configurations, calibrates and evaluates on held-out records, and saves diagnostic plots. Educational use only; not a clinical tool.

## 1. Imports and run configuration

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.calibration import CalibrationDisplay
from sentinel_health.ckd_data import DATA_PATH, FEATURE_COLUMNS, NUMERIC_COLUMNS, TARGET, load_dataset, make_features, profile_dataset
from sentinel_health.ckd_training import ARTIFACTS, _candidates, run_training

DATA_PATH, ARTIFACTS

## 2. Load the ARFF data and inspect rows/columns

In [ ]:
df = load_dataset(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

## 3. Validate the target and profile data quality

In [ ]:
profile = profile_dataset(df)
print(json.dumps(profile, indent=2))

In [ ]:
target_counts = df[TARGET].value_counts().rename_axis('label').to_frame('records')
target_counts['share'] = target_counts.records / len(df)
target_counts

In [ ]:
missing = df[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
missing.style.format({'missing_rate':'{:.1%}'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
df[TARGET].value_counts().reindex(['ckd','notckd']).plot.bar(ax=axes[0], color=['#2B5F9E','#22806D'])
axes[0].set(title='Outcome distribution', xlabel='Class', ylabel='Records')
missing.head(15).sort_values('missing_rate').plot.barh(ax=axes[1], legend=False, color='#4A80B8')
axes[1].set(title='Highest missingness fields', xlabel='Missing fraction', ylabel='')
plt.tight_layout()

## 4. EDA: numeric labs and class association

Numeric distributions can overlap and have very different scales. These plots are descriptive, not clinical reference ranges.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16,10))
for ax, col in zip(axes.ravel(), NUMERIC_COLUMNS):
    for label, color in [('ckd','#2B5F9E'),('notckd','#22806D')]:
        values = df.loc[df[TARGET]==label, col].dropna()
        ax.hist(values, bins=16, alpha=.55, density=True, label=label, color=color)
    ax.set_title(col); ax.legend(fontsize=7)
for ax in axes.ravel()[len(NUMERIC_COLUMNS):]: ax.axis('off')
plt.tight_layout()

In [ ]:
corr = df[NUMERIC_COLUMNS].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9,7))
image=ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=70, ha='right')
ax.set_yticks(range(len(corr.columns)), corr.columns)
fig.colorbar(image, ax=ax, label='Pearson correlation'); ax.set_title('Numeric measurement correlation')
plt.tight_layout()

In [ ]:
categorical = [c for c in FEATURE_COLUMNS if c not in NUMERIC_COLUMNS]
for col in categorical[:6]:
    tab = pd.crosstab(df[col].fillna('Missing'), df[TARGET], normalize='index')
    display(tab.style.format('{:.1%}').set_caption(f'{col}: class proportions by category'))

## 5. Outlier review

IQR flags identify values for review. They do not cause row deletion: extreme lab values can be real and informative. Scaling is fitted inside each model fold.

In [ ]:
outlier_rows=[]
for col in NUMERIC_COLUMNS:
    q1,q3=df[col].quantile([.25,.75]); iqr=q3-q1
    flagged=(df[col]<q1-1.5*iqr)|(df[col]>q3+1.5*iqr)
    outlier_rows.append({'feature':col,'iqr_flagged':int(flagged.sum()),'rate':float(flagged.mean())})
outlier_report=pd.DataFrame(outlier_rows).sort_values('rate',ascending=False)
outlier_report

In [ ]:
outlier_report.sort_values('rate').plot.barh(x='feature',y='rate',legend=False,color='#B7791F',figsize=(8,5),title='IQR-flagged rate (review only)')
plt.xlabel('Fraction of non-missing values flagged'); plt.tight_layout()

## 6. Preprocessing and conservative feature engineering

The ARFF source uses `?`/blank markers and has one malformed row with an extra empty cell; the loader normalizes both while verifying the 25-column schema and 400 records. Numeric fields use median imputation, missing indicators and configurable scaling. Nominal fields use mode or `Unknown` imputation and one-hot encoding. The derived urea-to-creatinine ratio is guarded against zero creatinine, and a numeric-lab missing count is added. No target-derived or patient-ID fields are used.

In [ ]:
X = make_features(df)
y = df['target'].astype('int8')
print('Feature shape:',X.shape,'| positive fraction:',f'{y.mean():.1%}')
print('Numeric:',list(X.select_dtypes(include=np.number).columns))
print('Categorical:',list(X.select_dtypes(exclude=np.number).columns))
X.head()

## 7. Stratified split and duplicate protection

There is no patient ID in the supplied file. Exact duplicate feature rows are grouped so they cannot cross training, validation, test, or CV folds. The final test partition is isolated before comparing candidates.

In [ ]:
duplicate_group=pd.util.hash_pandas_object(df[FEATURE_COLUMNS].astype('string'),index=False).astype(str).to_numpy()
indices=np.arange(len(df))
outer=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
dev_local,test_local=next(outer.split(X,y,duplicate_group))
dev=indices[dev_local]; test=indices[test_local]
inner=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=43)
train_local,val_local=next(inner.split(X.iloc[dev],y.iloc[dev],duplicate_group[dev]))
train=dev[train_local]; validation=dev[val_local]
summary=[]
for name,idx in [('train',train),('validation',validation),('test',test)]:
    summary.append({'partition':name,'rows':len(idx),'ckd_count':int(y.iloc[idx].sum()),'ckd_rate':float(y.iloc[idx].mean()),'duplicate_groups':len(set(duplicate_group[idx]))})
pd.DataFrame(summary)

In [ ]:
groups={name:set(duplicate_group[idx]) for name,idx in [('train',train),('validation',validation),('test',test)]}
assert not(groups['train']&groups['validation'] or groups['train']&groups['test'] or groups['validation']&groups['test'])
print('No exact duplicate feature row crosses partitions.')

## 8. Thirty model/preprocessing combinations

The grid compares logistic regression, SVM, K-nearest neighbors, random forest, ExtraTrees, and histogram gradient boosting. Configurations vary regularization, class weighting, scaling, neighbor counts, tree depth/leaf size, boosting learning rate, and regularization. Candidate ranking uses five-fold stratified CV inside the training partition. AP is interpreted relative to prevalence, while ROC-AUC, balanced accuracy, precision, and recall are also reported.

In [ ]:
candidates = _candidates(make_features(df.iloc[train]))
print('Configured candidates:',len(candidates))
pd.DataFrame([{'candidate':name,'estimator':type(model.named_steps['model']).__name__} for name,model in candidates])

## 9. Train, calibrate, select threshold, and evaluate holdout

This cell runs all 30 candidates, selects the winner by cross-validation ROC-AUC (ties by AP), calibrates with cross-validation on training data, chooses thresholds using validation only, and scores the held-out partition once. It writes artifacts under `artifacts/ckd/`.

In [ ]:
started=time.perf_counter()
result=run_training(DATA_PATH,ARTIFACTS,cv_folds=5)
report=result['report']
print(f"Completed in {(time.perf_counter()-started)/60:.2f} minutes")
print('Selected candidate:',report['selected_model'])
print('Model version:',report['model_version'])

In [ ]:
cv_results=pd.DataFrame(report['cv_results']).sort_values('mean_cv_roc_auc',ascending=False)
cv_results

In [ ]:
cv_results.sort_values('mean_cv_roc_auc').plot.barh(x='candidate',y='mean_cv_roc_auc',xerr='std_cv_roc_auc',legend=False,color='#3973AC',figsize=(10,10),title='30 candidates ? stratified CV ROC-AUC')
plt.xlabel('Mean ROC-AUC'); plt.tight_layout()

## 10. Holdout metrics and operating threshold tradeoff

The threshold is selected on validation. The 80%-recall policy is an alternative threshold; holdout recall can differ. Average precision is high in part because 62.5% of the source rows are positive, so compare it with the positive prevalence and read it alongside ROC-AUC and calibration.

In [ ]:
metrics=report['test_metrics']
pd.Series(metrics,name='held-out test').to_frame()

In [ ]:
print('Validation max-F1 policy:',json.dumps(report['validation_policy'],indent=2))
print('Validation 80%-recall policy:',json.dumps(report['validation_recall_80_policy'],indent=2))
print('Holdout at high-recall policy:',json.dumps(report['test_metrics_at_validation_recall_80_threshold'],indent=2))
print('Prevalence baseline:',json.dumps(report['dummy_baseline'],indent=2))

In [ ]:
pred=pd.read_csv(ARTIFACTS/'holdout_predictions.csv')
actual=pred.actual.to_numpy(); probability=pred.probability.to_numpy(); threshold=report['validation_policy']['threshold']
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
ConfusionMatrixDisplay.from_predictions(actual,probability>=threshold,display_labels=['notckd','ckd'],cmap='Blues',ax=axes[0],colorbar=False)
PrecisionRecallDisplay.from_predictions(actual,probability,ax=axes[1],name='CKD classifier')
RocCurveDisplay.from_predictions(actual,probability,ax=axes[2],name='CKD classifier')
axes[0].set_title('Holdout confusion matrix'); axes[1].set_title('Holdout precision-recall'); axes[2].set_title('Holdout ROC')
plt.tight_layout()

In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
CalibrationDisplay.from_predictions(actual,probability,n_bins=5,strategy='quantile',ax=ax)
ax.set_title('Holdout probability calibration'); plt.tight_layout()

## 11. Explainability, drift and inference observability

In [ ]:
importance=pd.read_csv(ARTIFACTS/'feature_importance.csv').head(15).sort_values('importance')
importance.plot.barh(x='feature',y='importance',xerr='importance_std',legend=False,color='#3973AC',figsize=(8,5),title='Validation permutation importance')
plt.tight_layout()

In [ ]:
pd.read_csv(ARTIFACTS/'feature_drift.csv').head(20)

In [ ]:
print('Offline single-row inference latency (ms):',report['offline_inference_latency'])
print('CV duration (s):',round(report['duration_seconds'],2))

## 12. Saved artifacts and limitations

The model, comparison table, data quality, IQR review, feature importance, drift, holdout predictions, PR/ROC/confusion/calibration figures, registry and model card are saved under `artifacts/ckd/`. The dataset has only 400 records and is from one short historical collection. Very high internal metrics do not establish external or prospective performance. Inputs include measurements and comorbidities that may be closely tied to diagnosis; define prediction timing before any real-world interpretation. This project is an educational classifier, not a diagnostic tool.

In [ ]:
print('Artifacts:',ARTIFACTS.resolve())
print('Holdout ROC-AUC:',f"{metrics['roc_auc']:.4f}",'| AP:',f"{metrics['average_precision']:.4f}")
print('Precision / recall / F1:',*(f"{metrics[k]:.4f}" for k in ['precision','recall','f1']))